In [ ]:
# 1. Instal·lació de dependències per a Google Colab
%%shell
wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list
apt-get update
apt-get install -y google-chrome-stable
pip install selenium==4.18.1 webdriver-manager
google-chrome-stable --version

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import quote
import glob
import os
import time
import shutil
import pandas as pd
from google.colab import files

In [ ]:
def web_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("--window-size=1920,1200")
    
    # Carpeta per a les descàrregues automàtiques a Colab
    prefs = {"download.default_directory": "/content/"}
    options.add_experimental_option("prefs", prefs)
    
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

In [ ]:
def descarrega_xlsx(nou_nom):
    # Busca el fitxer més recent amb format brand_db_export
    xlsx_files = glob.glob('/content/brand_db_export*.xlsx')
    if not xlsx_files:
         xlsx_files = [f for f in glob.glob('/content/*.xlsx') if f not in ['/content/entitats.xlsx', '/content/log_results.xlsx']]
    
    if not xlsx_files: return False

    xlsx_files.sort(key=os.path.getmtime, reverse=True)
    ultim_arxiu = xlsx_files[0]

    wipo_folder = '/content/wipo'
    if not os.path.exists(wipo_folder): os.makedirs(wipo_folder)
    
    safe_name = str(nou_nom).replace('/', '_').replace('\\', '_').replace(':', '_').replace('?', '_').replace('*', '_')
    nou_nom_complet = os.path.join(wipo_folder, safe_name + ".xlsx")
    
    try:
        if os.path.exists(nou_nom_complet): os.remove(nou_nom_complet)
        shutil.move(ultim_arxiu, nou_nom_complet)
        return True
    except Exception as e:
        print(f"      Error movent fitxer: {e}")
        return False

In [ ]:
def titular(driver, codi):
  print(f"Processant: {codi}...")
  
  # 1. Carreguem la pàgina base
  driver.get("https://branddb.wipo.int/en/advancedsearch")
  
  try:
    # 2. Esperem que el quadre de cerca estigui actiu
    wait = WebDriverWait(driver, 35)
    
    # Usem un selector d'input de cerca més genèric per trobar el quadre de text principal
    # La web de la WIPO sol tenir algun input inicialment actiu
    search_input = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input[type='text'], input[placeholder*='search'], input[role='combobox']")))
    
    # 3. Usem el PREFIX de cerca: applicant:"Nom de l'entitat"
    # Això és la manera més segura que la WIPO ens entengui des de qualsevol quadre de text
    search_cmd = f'applicant:"{codi}"'
    
    search_input.clear()
    search_input.send_keys(search_cmd)
    time.sleep(1)
    search_input.send_keys(Keys.ENTER)

    # 4. Verifiquem resultats
    download_xpath = '//span[text()="Download results"]'
    
    # Esperem fins que bé hi hagi el botó o bé el missatge de zero resultats
    # Ens fixem en la cadena de text 'Displaying 0-0' que indica que no hi ha resultats
    wait.until(lambda d: d.find_elements(By.XPATH, download_xpath) or 
                         "Displaying 0-0" in d.page_source or 
                         "No records found" in d.page_source)
    
    if not driver.find_elements(By.XPATH, download_xpath):
        print(f"   ! Cap marca trobada (applicant per prefix) per a {codi}")
        return codi, "No trobat"

    # 5. Clica el botó de descàrrega
    btn = driver.find_element(By.XPATH, download_xpath)
    driver.execute_script("arguments[0].scrollIntoView();", btn)
    time.sleep(1)
    btn.click()
    
    # 6. Seleccionem format EXCEL
    excel_opt = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable((By.XPATH, '//li//span[text()="EXCEL"]'))
    )
    excel_opt.click()

    # Temps d'espera per a completar la descàrrega
    time.sleep(12)

    if descarrega_xlsx(codi):
        print(f"   OK: Descarregada informació de {codi}")
        return codi, "1"
    else:
        print(f"   ? Problema al detectar el fitxer de {codi}")
        return codi, "Error fitxer"

  except TimeoutException:
    print(f"   Timeout intentant buscar: {codi}")
    return codi, "Timeout"
  except Exception as e:
    print(f"   Error en processar {codi}: {type(e).__name__}")
    return codi, f"Error {type(e).__name__}"

In [ ]:
# Carreguem les dades inicials (entitats.xlsx)
if os.path.exists("entitats.xlsx"):
    df = pd.read_excel("entitats.xlsx")
    # Neteja i filtrat dels noms exclusius
    entities = df['Titular (entitat)'].apply(lambda x: str(x).rsplit(",", 1)[0].rsplit(' (', 1)[0].strip()).unique()

    log = []
    print(f"Iniciant driver (V4 - Prefix logic) per a {len(entities)} entitats...")
    driver = web_driver()
    
    try:
        for ent in entities:
            if ent and ent != 'nan':
                entitat, status = titular(driver, ent)
                log.append([entitat, status])
    finally:
        driver.quit()
        pd.DataFrame(log, columns=["Entitat", "Estat"]).to_excel("log_results_v4.xlsx", index=False)
        print("\nExecució finalitzada. Resultats guardats a log_results_v4.xlsx")
else:
    print("Atenció: Puja el fitxer 'entitats.xlsx' a Colab per començar.")

In [ ]:
# Comprimir la carpeta de resultats i baixar-la
if os.path.exists('/content/wipo'):
    if os.path.exists('marques_wipo_export_v4.zip'): os.remove('marques_wipo_export_v4.zip')
    shutil.make_archive('marques_wipo_export_v4', 'zip', '/content/wipo')
    files.download('marques_wipo_export_v4.zip')
    print("L'enllaç per descarregar el ZIP de resultats s'està preparant...")